<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-07/AI_Modul_7.1_Praktikum_Linear_Regression.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 7.1: Praktikum Pemodelan Regresi Linier (Supervised Learning)

**Mata Kuliah:** Kecerdasan Buatan & Pembelajaran Mesin Terapan  
**Institusi:** Institut Pertanian Stiper (INSTIPER) Yogyakarta  
**Studi Kasus:** Estimasi Produktivitas Tandan Buah Segar (TBS) Kelapa Sawit Berdasarkan Variabel Agroklimat & Pemupukan

---

### Capaian Pembelajaran Praktikum
1. Mampu mengimplementasikan formulasi analitik Ordinary Least Squares (OLS) secara manual menggunakan NumPy.
2. Mampu membangun model Regresi Linier Terapan menggunakan pustaka Scikit-Learn.
3. Mampu melakukan evaluasi kinerja model regresi menggunakan metrik MAE, RMSE, dan $R^2$.
4. Mampu mendeteksi pelanggaran asumsi klasik Gauss-Markov (uji multikolinearitas VIF dan analisis grafik residual).

## 1. Import Pustaka Komputasi & Pengaturan Lingkungan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka berhasil diimpor dengan sukses!")

## 2. Pembangkitan Dataset Agroklimat & Produktivitas Kelapa Sawit

Dataset ini memodelkan 150 blok perkebunan sawit mandiri dengan variabel:
- `Curah_Hujan_mm`: Curah hujan tahunan (mm/tahun)
- `Pupuk_NPK_kg`: Dosis pupuk NPK (kg/pokok/tahun)
- `Radiasi_Jam`: Jam penyinaran matahari rerata harian (jam/hari)
- `Defisit_Air_mm`: Akumulasi defisit air tahunan (mm)
- `Yield_TBS_TonHa`: Produktivitas panen TBS (ton/ha/tahun)

In [ ]:
np.random.seed(42)
n_samples = 150

# Variabel penjelas (fitur agronomis)
curah_hujan = np.random.uniform(1600, 2900, n_samples)
pupuk_npk = np.random.uniform(4.0, 9.5, n_samples)
radiasi_surya = np.random.uniform(4.5, 7.8, n_samples)
defisit_air = np.maximum(0, 2350 - curah_hujan) * np.random.uniform(0.18, 0.28, n_samples)

# Relasi riil biologis dengan galat acak (noise)
noise = np.random.normal(0, 1.15, n_samples)
yield_tbs = 8.2 + 0.0038 * curah_hujan + 1.35 * pupuk_npk + 0.78 * radiasi_surya - 0.016 * defisit_air + noise

df_sawit = pd.DataFrame({
    'Curah_Hujan_mm': np.round(curah_hujan, 1),
    'Pupuk_NPK_kg': np.round(pupuk_npk, 2),
    'Radiasi_Jam': np.round(radiasi_surya, 2),
    'Defisit_Air_mm': np.round(defisit_air, 1),
    'Yield_TBS_TonHa': np.round(yield_tbs, 2)
})

print("Tampilan 5 Sampel Data Teratas:")
display(df_sawit.head())

## 3. Eksplorasi Data & Matriks Korelasi Pearson

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_sawit.corr()
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt=".3f", ax=ax, linewidths=0.5)
ax.set_title("Matriks Korelasi Pearson Variabel Agronomi & Yield TBS", fontsize=12, fontweight='bold', pad=12)
plt.show()

## 4. Pembagian Partisi Dataset (Training & Testing)

In [ ]:
fitur = ['Curah_Hujan_mm', 'Pupuk_NPK_kg', 'Radiasi_Jam', 'Defisit_Air_mm']
target = 'Yield_TBS_TonHa'

X = df_sawit[fitur]
y = df_sawit[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensi Training Set : {X_train.shape}")
print(f"Dimensi Testing Set  : {X_test.shape}")

## 5. Implementasi OLS Aljabar Linier Murni (NumPy)

Formula analitik persamaan normal OLS:
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Untuk stabilitas numerik, kita menyelesaikan sistem persamaan $\mathbf{X}^T \mathbf{X} \hat{\boldsymbol{\beta}} = \mathbf{X}^T \mathbf{y}$ menggunakan fungsi `np.linalg.solve`.

In [ ]:
# 1. Tambahkan kolom bias (angka 1) ke matriks desain X
X_mat_train = np.column_stack([np.ones(X_train.shape[0]), X_train.values])
y_vec_train = y_train.values

# 2. Selesaikan persamaan normal OLS
beta_manual = np.linalg.solve(X_mat_train.T @ X_mat_train, X_mat_train.T @ y_vec_train)

print("=== ESTIMASI PARAMETER OLS MANUAL (NUMPY) ===")
print(f"Intercept (beta_0)   : {beta_manual[0]:.4f}")
for f, b in zip(fitur, beta_manual[1:]):
    print(f"Slope ({f:<15}) : {b:+.4f}")

## 6. Pemodelan Menggunakan Scikit-Learn & Verifikasi Ekuivalensi

In [ ]:
model_sk = LinearRegression(fit_intercept=True)
model_sk.fit(X_train, y_train)

print("=== ESTIMASI PARAMETER SCIKIT-LEARN ===")
print(f"Intercept (model.intercept_) : {model_sk.intercept_:.4f}")
for f, b in zip(fitur, model_sk.coef_):
    print(f"Slope ({f:<15})         : {b:+.4f}")

# Verifikasi matematis ekuivalensi kedua pendekatan
np.testing.assert_almost_equal(beta_manual[0], model_sk.intercept_, decimal=5)
np.testing.assert_almost_equal(beta_manual[1:], model_sk.coef_, decimal=5)
print("\n[VERIFIED] Hasil OLS Manual NumPy dan Scikit-Learn identik 100%!")

## 7. Evaluasi Kinerja Model Prediksi (Testing Set)

In [ ]:
y_pred = model_sk.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)

print("=== METRIK EVALUASI REGRESI LINIER SAWIT ===")
print(f"Mean Absolute Error (MAE)        : {mae:.3f} Ton/Ha")
print(f"Mean Squared Error (MSE)         : {mse:.3f} (Ton/Ha)^2")
print(f"Root Mean Squared Error (RMSE)   : {rmse:.3f} Ton/Ha")
print(f"Koefisien Determinasi (R^2)      : {r2:.4f} ({r2*100:.2f}%)")
print(f"Adjusted R^2                     : {adj_r2:.4f}")

## 8. Diagnostik Asumsi Klasik Gauss-Markov

Kita memvisualisasikan 4 grafik diagnostik residual:
1. **Residuals vs Fitted Values** (Linearitas & Homoskedastisitas)
2. **Normal Q-Q Plot** (Normalitas galat)
3. **Scale-Location Plot** (Dispersi varians galat)
4. **Cook's Distance** (Identifikasi data pengungkit / outlier berpengaruh)

In [ ]:
residuals_train = y_train - model_sk.predict(X_train)
y_fitted = model_sk.predict(X_train)
std_res = (residuals_train - residuals_train.mean()) / residuals_train.std()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
ax1 = axes[0, 0]
ax1.scatter(y_fitted, residuals_train, color='#1f77b4', edgecolors='k', alpha=0.7)
ax1.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax1.set_title("A. Residual vs Fitted (Linearitas & Homoskedastisitas)", fontweight='bold')
ax1.set_xlabel("Prediksi Nilai Panen (Ton/Ha)")
ax1.set_ylabel("Residual")

# 2. Normal Q-Q Plot
ax2 = axes[0, 1]
stats.probplot(std_res, dist="norm", plot=ax2)
ax2.set_title("B. Normal Q-Q Plot (Normalitas Residual)", fontweight='bold')
ax2.get_lines()[0].set_color('#2ca02c')
ax2.get_lines()[0].set_markersize(6)
ax2.get_lines()[1].set_color('red')

# 3. Scale-Location Plot
ax3 = axes[1, 0]
ax3.scatter(y_fitted, np.sqrt(np.abs(std_res)), color='#ff7f0e', edgecolors='k', alpha=0.7)
ax3.set_title("C. Scale-Location Plot (Konstansi Varians Galat)", fontweight='bold')
ax3.set_xlabel("Prediksi Nilai Panen (Ton/Ha)")
ax3.set_ylabel("Akar Kuadrat Standar Residual Mutlak")

# 4. Cook's Distance
ax4 = axes[1, 1]
# Matriks Hat: H = X(X^T X)^-1 X^T
H = X_mat_train @ np.linalg.inv(X_mat_train.T @ X_mat_train) @ X_mat_train.T
leverage = np.diag(H)
mse_t = np.mean(residuals_train**2)
p_dim = X_mat_train.shape[1]
cooks_d = (residuals_train**2 / (p_dim * mse_t)) * (leverage / (1 - leverage)**2)

ax4.stem(range(len(cooks_d)), cooks_d, linefmt='b-', markerfmt='bo', basefmt='r-')
threshold = 4 / len(cooks_d)
ax4.axhline(threshold, color='red', linestyle='--', label=f'Threshold 4/n ({threshold:.3f})')
ax4.set_title("D. Cook's Distance (Deteksi Outlier Berpengaruh)", fontweight='bold')
ax4.set_xlabel("Indeks Sampel Blok Kebun")
ax4.set_ylabel("Cook's Distance")
ax4.legend()

plt.tight_layout()
plt.show()

## 9. Evaluasi Multikolinearitas (Variance Inflation Factor / VIF)

In [ ]:
X_vif = X_train.copy()
X_vif.insert(0, 'Konstanta', 1.0)

vif_df = pd.DataFrame({
    'Variabel': X_train.columns,
    'Nilai_VIF': [variance_inflation_factor(X_vif.values, i) for i in range(1, X_vif.shape[1])]
})
vif_df['Status'] = np.where(vif_df['Nilai_VIF'] < 5, 'Aman (Bebas Multikolinearitas)', 'Perlu Diinvestigasi')
display(vif_df)

## 10. Tugas Mandiri & Eksplorasi Mahasiswa

1. Lakukan simulasi skenario panen: Berapakah taksiran produksi TBS (Ton/Ha) jika suatu blok kebun baru memiliki data agroklimat berikut?
   - Curah Hujan = 2.200 mm/tahun
   - Dosis Pupuk NPK = 7.5 kg/pokok/tahun
   - Radiasi Surya = 6.2 jam/hari
   - Defisit Air = 15.0 mm/tahun
2. Masukkan fitur baru bernama `Pupuk_Urea_kg` yang sengaja dibuat berkorelasi 0.98 dengan `Pupuk_NPK_kg`. Amati perubahan nilai VIF dan tanda koefisien regresi!
3. Tuliskan kesimpulan analitis Anda mengenai interpretasi koefisien regresi pada pemodelan kelapa sawit tersebut.